In [ ]:
import pandas as pd
import os
import subprocess
import shutil
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests
import stat
import time

# === CONFIGURATION ===
MAX_PROJECTS = 2482
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
headers = {'Authorization': f'token {GITHUB_TOKEN}'}

if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

START_NUMBER = int(os.getenv("START_NUMBER"))
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = Path(r"D:\Android_Mobile_App\AndroidProject_4th\Android_Repos_MultiRange.csv")
base_dir = Path(r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_path = base_dir / "8.2-Project_Metadata.csv"
config_location_csv = base_dir / "Config_Location.csv"
log_path = base_dir / "repo_processing.log"
failed_projects = []



# === ENSURE ALL FOLDERS EXIST ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir, cloned_sample_dir]:
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['clone_url'].notna()]
df['github_url'] = df['clone_url'].astype(str).str.strip()
df = df[df['clone_url'].str.startswith("https://")]
df[['clone_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# === LOAD EXISTING CONFIG LOCATIONS IF RESUMING ===
if config_location_csv.exists():
    config_locations_df = pd.read_csv(config_location_csv)
else:
    config_locations_df = pd.DataFrame(columns=["repo_name", "config_file_path", "file_type"])

#log_message for warnings and errors
def log_message(message):
    print(message)
    with open(log_path, "a", encoding="utf-8") as log_file:
        log_file.write(message + "\n")


def check_rate_limit():
    response = requests.get("https://api.github.com/rate_limit", headers=headers)
    if response.status_code == 200:
        remaining = response.json()['rate']['remaining']
        reset_time = response.json()['rate']['reset']
        if remaining < 50:
            wait_seconds = reset_time - int(time.time())
            if wait_seconds > 0:
                log_message(f"⏳ Rate limit low. Sleeping for {wait_seconds} seconds...")
                time.sleep(wait_seconds + 5)  # Add buffer
        return remaining
    else:
        log_message("⚠️ Could not check rate limit.")
        return None

#========= Commit Hist from GitHub API
def extract_commit_metadata_api(owner, repo, max_commits=500):
    url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    commits = []
    page = 1
    check_rate_limit()

    while len(commits) < max_commits:
        paged_url = f"{url}?per_page=100&page={page}"
        response = requests.get(paged_url, headers=headers)
        if response.status_code != 200:
            log_message(f"❌ Error fetching {owner}/{repo} commits: {response.status_code}")
            

            break

        data = response.json()
        if not data:
            break

        for commit in data:
            commit_data = commit.get("commit", {})
            author = commit_data.get("author", {})
            commits.append({
                "sha": commit.get("sha"),
                "author_name": author.get("name"),
                "author_email": author.get("email"),
                "date": author.get("date"),
                "message": commit_data.get("message"),
                "html_url": commit.get("html_url")
            })

            if len(commits) >= max_commits:
                break

        page += 1

    if commits:
        
        df_commits = pd.DataFrame(commits)
        filename = f"{owner}.{repo}.commits.csv"
        df_commits.to_csv(commits_dir/filename, index=False)
        print(f"✅ API commit data saved for {owner}/{repo}")
    else:
        log_message(f"⚠️ No commits found for {owner}/{repo} via API")
        failed_projects.append(repo_name)

        





# # === COMMIT METADATA EXTRACTION FUNCTION ===
# def extract_commit_metadata(repo_path, output_path):
#     try:
#         cmd_hashes = ["git", "-C", str(repo_path), "log", "--pretty=format:%H"]
#         result_hashes = subprocess.run(cmd_hashes, capture_output=True, text=True, check=True)
#         commit_hashes = result_hashes.stdout.strip().split("\n")

#         rows = []
#         for commit in commit_hashes:
#             cmd_metadata = ["git", "-C", str(repo_path), "show", "--quiet",
#                             f"--pretty=format:%H|%an|%ae|%ad|%s", "--date=iso", commit]
#             result_metadata = subprocess.run(cmd_metadata, capture_output=True, text=True, check=True)
#             parts = result_metadata.stdout.strip().split("|", maxsplit=4)
#             if len(parts) < 5:
#                 continue

#             cmd_files = ["git", "-C", str(repo_path), "show", "--name-only", "--pretty=format:", commit]
#             result_files = subprocess.run(cmd_files, capture_output=True, text=True, check=True)
#             changed_files = result_files.stdout.strip().split("\n")
#             changed_files = [f.strip() for f in changed_files if f.strip()]

#             count_androidTest = sum("androidTest" in f for f in changed_files)
#             count_github_workflows = sum(".github/workflows" in f for f in changed_files)
#             count_gradle = sum("build.gradle" in f for f in changed_files)

#             rows.append({
#                 "commit_hash": parts[0],
#                 "author_name": parts[1],
#                 "author_email": parts[2],
#                 "commit_date": parts[3],
#                 "commit_message": parts[4],
#                 "touches_androidTest": count_androidTest > 0,
#                 "count_androidTest": count_androidTest,
#                 "touches_github_workflows": count_github_workflows > 0,
#                 "count_github_workflows": count_github_workflows,
#                 "touches_gradle": count_gradle > 0,
#                 "count_gradle": count_gradle
#             })

#         if rows:
#             df = pd.DataFrame(rows)
#             output_path.mkdir(parents=True, exist_ok=True)
#             df.to_csv(output_path / "contributors_commits.csv", index=False)
#             print(f"✅ Saved commit metadata for {repo_path.name}")
#         else:
#             print(f"⚠️ No commit data for {repo_path.name}")

#     except subprocess.CalledProcessError as e:
#         print(f"❌ Failed to extract commit data for {repo_path.name}: {e}")

# === FUNCTION TO HANDLE READ-ONLY FILES ===
def force_remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)

# === FUNCTION TO GET COUNT FROM GITHUB API ===
def get_count(api_url, headers):
    check_rate_limit()

    try:
        r = requests.get(api_url, headers=headers, params={"per_page": 1})
        if 'Link' in r.headers:
            return int(r.headers['Link'].split(',')[0].split('page=')[-1].split('>')[0])
        return len(r.json())
    except:
        return 0

# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.iloc[i]['github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}.{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        #subprocess.run(['git', 'clone', url, str(repo_path)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', url, str(repo_path)],
               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        print("✅ Clone complete")
    except Exception as e:
        #print(f"❌ Clone failed for {repo_name}: {e}")
        log_message(f"❌ Clone failed for {repo_name}: {e}")
        failed_projects.append(repo_name)

        continue

        # === Detect and checkout default branch from GitHub API ===
    try:
        base_api = f"https://api.github.com/repos/{username}/{project}"
        headers = {'Authorization': f'token {GITHUB_TOKEN}'}
        check_rate_limit()
        r_branch = requests.get(base_api, headers=headers, timeout=15)
        if r_branch.status_code == 200:
            default_branch = r_branch.json().get('default_branch', 'main')
            subprocess.run(["git", "-C", str(repo_path), "checkout", default_branch],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"📌 Checked out default branch: {default_branch}")
        else:
            print(f"⚠️ Could not detect default branch for {repo_name}, using current HEAD")
    except Exception as e:
        log_message(f"⚠️ Failed to checkout default branch for {repo_name}: {e}")
        #failed_projects.append(repo_name)



        # === Commit metadata via GitHub API ===
    #metadata_dir = git_metadata_dir / repo_name
    extract_commit_metadata_api(username, project)



    # === Check commit count ===
    #metadata_dir = git_metadata_dir / repo_name
    #==== this block tries to get Commit hist from Full Clone with does not work for now
    #try:
    #    result = subprocess.run(['git', '-C', str(repo_path), 'rev-list', '--count', 'HEAD'], capture_output=True, text=True, check=True)
    #    local_commit_count = int(result.stdout.strip())
    #except subprocess.CalledProcessError:
    #    local_commit_count = 0
    #    print(f"⚠️ Could not get commit count for {repo_name}")

    #if local_commit_count > 0:
    #    extract_commit_metadata(repo_path, metadata_dir)
    #else:
    #    print(f"⚠️ No commits to extract for {repo_name}")

    # === Scan and copy config/build files ===
    ci_keywords = ['ci', 'build', 'test', 'workflow', 'pipeline', 'instrumentation']
    config_files_found = []

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_lower = file.lower()
            file_path = Path(root) / file
            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")

            try:
                should_copy = False
                file_type = file_lower.split('.')[-1]

                  # === Determine CI Platform ===
                ci_platform = "Other"
                rel_path_lower = rel_path.lower()

                if ".github/workflows" in rel_path_lower:
                    ci_platform = "GitHub"
                elif "circleci" in rel_path_lower:
                    ci_platform = "CircleCI"
                elif "gitlab-ci" in rel_path_lower or "gitlab" in rel_path_lower:
                    ci_platform = "GitLab CI"
                elif "travis" in rel_path_lower:
                    ci_platform = "Travis CI"
                elif "bitrise" in rel_path_lower:
                    ci_platform = "Bitrise"


                # === Determine if file qualifies as config ===
                if file_lower.endswith(('.yml', '.yaml')):
                    should_copy = True

                elif file_lower.endswith(('build.gradle', 'build.gradle.kts')): #['test', 'instrumentation']
                    # Store ALL build files unconditionally
                    should_copy = True

                elif file_lower.endswith(('.json', '.sh')):
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read().lower()
                        if any(keyword in content for keyword in ci_keywords):
                            should_copy = True

                # === If it qualifies, copy to Config Files with custom name ===
                if should_copy:
                    # Build the flat filename
                    rel_parts = rel_path.replace("/", ".").replace("\\", ".")
                    flat_filename = f"{username}.{project}.{ci_platform}.{file}"

                    # Save to build folder
                    if file_lower.endswith('build.gradle'):
                        destination_path = build_info_dir / flat_filename
                    else:
                        destination_path = yml_output_dir / flat_filename
                    shutil.copy2(file_path, destination_path)
                    
                    config_files_found.append({
                        "repo_name": repo_name,
                        "config_file_path": flat_filename,
                        "file_type": file_type
                    })
                    
            except Exception as e:
                log_message(f"⚠️ Could not process or copy {rel_path} in {repo_name}: {e}")
                #log_message(f"❌ Could not process or copy {repo_name}: {e}")



    if config_files_found:
        config_locations_df = pd.concat([config_locations_df, pd.DataFrame(config_files_found)], ignore_index=True)

    # === Fetch and save metadata + contributors ===
    try:
        headers = {'Authorization': f'token {GITHUB_TOKEN}'}
        base_api = f"https://api.github.com/repos/{username}/{project}"

        check_rate_limit()
        r = requests.get(base_api, headers=headers, timeout=30)
        data = r.json()

                # === Count commits and unique contributors from CSV ===
        commit_csv_path = commits_dir / f"{username}.{project}.commits.csv"
        if commit_csv_path.exists():
            try:
                df_commits = pd.read_csv(commit_csv_path)
                total_commits = len(df_commits)
                unique_contributors = df_commits['author_name'].nunique()
            except Exception as e:
                log_message(f"⚠️ Could not read commit CSV for {repo_name}: {e}")
                total_commits = None
                unique_contributors = None
        else:
            total_commits = None
            unique_contributors = None

        metadata_row = {
            'project_name': project,
            'repo_name': repo_name,
            'full_name': data.get('full_name'),
            'description': data.get('description'),
            'language': data.get('language'),
            'license': data.get('license', {}).get('name') if data.get('license') else None,
            'created_at': data.get('created_at'),
            'updated_at': data.get('updated_at'),
            'last_commit_date': data.get('pushed_at'),
            'stars': data.get('stargazers_count'),
            'forks': data.get('forks_count'),
            'watchers': data.get('watchers_count'),
            'open_issues': data.get('open_issues_count'),
            'contributors': get_count(f"{base_api}/contributors", headers),
            'pull_requests': get_count(f"{base_api}/pulls?state=all", headers),
            'commits_GitAPI': total_commits,
            'contributors_with_commit': unique_contributors,
            'size': data.get('size')
        }

        metadata_df = pd.DataFrame([metadata_row])
        if metadata_path.exists():
            metadata_df.to_csv(metadata_path, mode='a', header=False, index=False)
        else:
            metadata_df.to_csv(metadata_path, mode='w', header=True, index=False)
        print("📜 Metadata saved")

        # === Save contributor names ===
        contrib_url = f"{base_api}/contributors"
        check_rate_limit()

        r_contrib = requests.get(contrib_url, headers=headers, timeout=30)
        if r_contrib.status_code == 200:
            contributor_logins = [c['login'] for c in r_contrib.json()]
            contributors_text = "\n".join(contributor_logins)
            # === Save contributors as single file in Config Files ===
            contributors_filename = f"{username}.{project}.contributors.txt"
            contributors_path = commits_dir / contributors_filename

            with open(contributors_path, "w", encoding="utf-8") as f:
                f.write(contributors_text)

            print(f"👥 Saved contributors to: {contributors_path.name}")

        else:
            log_message(f"⚠️ Failed to fetch contributors for {repo_name}: {r_contrib.status_code}")

    except Exception as e:
        log_message(f"⚠️ Metadata or contributors error for {repo_name}: {e}")
        failed_projects.append(repo_name)


    # === Move to Cloned_Sample or delete ===
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📆 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, onerror=force_remove_readonly)
            print(f"🕵️ Deleted cloned repo: {repo_name}")
            #print(f"🕵️ Single Search cloned repo: {repo_name}")
    except Exception as e:
        log_message(f"❌ Error handling repo folder for {repo_name}: {e}")

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))
    config_locations_df.drop_duplicates().to_csv(config_location_csv, index=False)

if failed_projects:
    failed_df = pd.DataFrame(failed_projects, columns=["repo_name"])
    failed_df.to_csv(base_dir / "Failed_Projects.csv", index=False)
    print(f"\n⚠️ {len(failed_projects)} projects failed. Saved to Failed_Projects.csv.")


print(f"\n✅ Process complete. Sampled: {len(sample_indices_to_keep)} | Total Processed: {len(df) - (START_NUMBER - 1)}")
print("\n✅ All selected repositories have been processed.")


🔁 Loaded SAMPLE_LIST from .env with 150 indices.

🔍 [1/2614] Processing 0000.JunkFood02.Seal...
✅ Clone complete
📌 Checked out default branch: main
✅ API commit data saved for JunkFood02/Seal
📜 Metadata saved
👥 Saved contributors to: JunkFood02.Seal.contributors.txt
🕵️ Deleted cloned repo: 0000.JunkFood02.Seal

🔍 [2/2614] Processing 0001.android.nowinandroid...
✅ Clone complete
📌 Checked out default branch: main
✅ API commit data saved for android/nowinandroid
📜 Metadata saved
👥 Saved contributors to: android.nowinandroid.contributors.txt
🕵️ Deleted cloned repo: 0001.android.nowinandroid

🔍 [3/2614] Processing 0002.square.picasso...
✅ Clone complete
📌 Checked out default branch: master
✅ API commit data saved for square/picasso
📜 Metadata saved
👥 Saved contributors to: square.picasso.contributors.txt
🕵️ Deleted cloned repo: 0002.square.picasso

🔍 [4/2614] Processing 0003.google.flexbox-layout...
✅ Clone complete
📌 Checked out default branch: main
✅ API commit data saved for google/flex